In [1]:
from pathlib import Path
import pandas as pd
import sys
import os

In [2]:
project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))

base_path = Path("../../../data/raw/wipo/ip_indicators").resolve()
staging_path = Path("../../../data/staging/wipo/ip_indicators").resolve()

base_path.mkdir(parents=True, exist_ok=True)
staging_path.mkdir(parents=True, exist_ok=True)

# Ensure pandas prints every single column
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)

print(f"Reading Raw from: {base_path}")
print(f"Saving Staging to: {staging_path}")

Reading Raw from: C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\raw\wipo\ip_indicators
Saving Staging to: C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\staging\wipo\ip_indicators


In [3]:
# Mapping short codes (patent -> PA)
type_map = {
    "industrial": "ID",
    "patent": "PA",
    "trademark": "TM"
}

In [4]:
# The specific files for "IP Flows by Class/Technology"
# Note: These filenames are estimated based on WIPO standard naming. 
# Ensure your actual .csv files match these names.
files = [
    # --- Patents (By Technology) ---
    "patent_4a - Patent publications by technology_Count by filling office and applicant's origin_1980_2024.csv",
    "patent_5 - Patent grants by technology_Count by filing office and applicant's origin_1980_2024.csv",

    # --- Trademarks (By Nice Class) ---
    "trademark_3a- Direct applications by class_Count by filing office and applicant's origin_1980_2024.csv",
    "trademark_3b- Applications via the Madrid system by class_Count by filing office and applicant's origin_1980_2024.csv",
    "trademark_4a- Registrations for direct applications by class_Count by filing office and applicant's origin_1980_2024.csv",
    "trademark_4b- Registrations for applications via the Madrid system by class_Count by filing office and applicant's origin_1980_2024.csv",

    # --- Industrial Designs (By Locarno Class) ---
    "industrial_3a- Direct applications by class_Count by filing office and applicant's origin_1980_2024.csv",
    "industrial_3b- Applications via the Hague system by class_Count by filing office and applicant's origin_1980_2024.csv",
    "industrial_4a- Registrations for direct applications by class_Count by filing office and applicant's origin_1980_2024.csv",
    "industrial_4b- Registrations for applications via the Hague system by class_Count by filing office and applicant's origin_1980_2024.csv"
]

In [5]:
for filename in files:
        path = base_path / filename
        if not path.exists():
            continue

        try:
            #  Add index_col=False to prevent the first column from becoming the index
            df = pd.read_csv(path, skiprows=6, index_col=False)

            # FIX COLUMN NAMES
            cols = list(df.columns)
            
            if len(cols) >= 4:
                cols[0] = "Office"
                cols[1] = "Office Code"
                cols[2] = "Origin"
                cols[3] = "class_code"
                
                df.columns = cols
            
     
            
            # Remove the phantom last column if it's all NaN and unnamed
            if "Unnamed" in str(df.columns[-1]):
                df = df.iloc[:, :-1]
            

            #Drop column named Office Code
            df.drop("Office Code", axis=1, inplace=True)

            #  EXPORT
            code_part = filename.split('-')[0].strip()
            parts = code_part.split('_')
            
            if len(parts) >= 2:
                short_cat = type_map.get(parts[0], parts[0])
                indicator_code = parts[1]
                out_name = f"stg_wipo__{short_cat}{indicator_code}.parquet"
                
                df.to_parquet(staging_path / out_name, index=False)
                print(f" Exported: {out_name} | Shape: {df.shape}")


        except Exception as e:
            print(f"[ERROR] {filename}: {e}")

 Exported: stg_wipo__PA4a.parquet | Shape: (106093, 48)
 Exported: stg_wipo__PA5.parquet | Shape: (76754, 48)
 Exported: stg_wipo__TM3a.parquet | Shape: (229726, 48)
 Exported: stg_wipo__TM3b.parquet | Shape: (276371, 48)
 Exported: stg_wipo__TM4a.parquet | Shape: (215496, 48)
 Exported: stg_wipo__TM4b.parquet | Shape: (280159, 48)
 Exported: stg_wipo__ID3a.parquet | Shape: (31700, 48)
 Exported: stg_wipo__ID3b.parquet | Shape: (38825, 48)
 Exported: stg_wipo__ID4a.parquet | Shape: (31656, 48)
 Exported: stg_wipo__ID4b.parquet | Shape: (34282, 48)
